# Phase 3: Joins

Enrich `trips_clean` with human-readable pickup/dropoff zone names from
`taxi_zone_lookup.csv` — a 9.3M-row fact table joined to a 265-row dimension
table. Classic setup for the broadcast-vs-shuffle join decision.

**Docs:** [Spark SQL Guide — Joins](https://spark.apache.org/docs/latest/sql-programming-guide.html#join-strategy-hints-for-sql-queries) | [`DataFrame.join` API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)

## Concept: shuffle (sort-merge) join vs. broadcast join

A **shuffle join** (Spark's default for large-large joins, aka sort-merge
join) repartitions *both* DataFrames across the network so matching keys land
on the same executor, then sorts and merges each partition. Cost scales with
both tables' size — expensive when one side is tiny, because you're still
paying to shuffle data that didn't need to move.

A **broadcast join** instead sends the *entire* small DataFrame to every
executor's memory (no shuffle needed for it), while the large DataFrame is
read and matched locally, partition by partition, with zero network shuffle
on the large side. Massive win when one side is small enough to fit in
memory — which 265 rows of zone lookups obviously is.

Spark **auto-broadcasts** any DataFrame under
`spark.sql.autoBroadcastJoinThreshold` (default 10MB) when it can estimate
the size (e.g. from Parquet metadata). For a CSV-sourced or already-filtered
DataFrame, Spark sometimes can't estimate size accurately — that's when you
use an explicit hint: `F.broadcast(df)`.

**Anti-pattern:** broadcasting a DataFrame that turns out to be large.
Every executor holds a full copy in memory — with a multi-GB "small" table
this causes OOM errors across the whole cluster, not just one task.


In [ ]:
# Setup: same read/cast/clean pipeline from notebooks 01-02, now imported
# from the dataforge_ai package instead of duplicated inline.
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from dataforge_ai import read_trips, clean_trips

spark = (
    SparkSession.builder
    .appName("DataForge-Phase3-Joins")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

RAW = "../data/raw"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"

def join_zones(trips: DataFrame, zones: DataFrame) -> DataFrame:
    pu = zones.alias("pu")
    do = zones.alias("do")
    return (
        trips
        .join(pu, trips.PULocationID == pu.LocationID, "left")
        .select(
            *trips.columns,
            F.col("pu.Zone").alias("pickup_zone"),
            F.col("pu.Borough").alias("pickup_borough"),
        )
        .join(do, trips.DOLocationID == do.LocationID, "left")
        .select(
            *trips.columns,
            "pickup_zone",
            "pickup_borough",
            F.col("do.Zone").alias("dropoff_zone"),
            F.col("do.Borough").alias("dropoff_borough"),
        )
    )

paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]
trips_clean = clean_trips(read_trips(spark, paths))

zones = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv(f"{RAW}/taxi_zone_lookup.csv")
)

print("trips_clean:", trips_clean.count(), "rows")
zones.show(5)

trips_clean: 9301798 rows
+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



## Small example: one join, two ways

Join `trips_clean` to `zones` on `PULocationID` — once with the default
(let Spark decide), once with an explicit broadcast hint — and compare the
physical plans with `.explain()`.

Docs: [`functions.broadcast`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html)


In [2]:
# Default: Spark decides (zones is tiny, so it'll likely auto-broadcast anyway
# since it's well under the 10MB autoBroadcastJoinThreshold).
default_join = trips_clean.join(zones, trips_clean.PULocationID == zones.LocationID)
default_join.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#217L], [cast(LocationID#1503 as bigint)], Inner, BuildRight, false
   :- SortAggregate(key=[DOLocationID#237L, tpep_dropoff_datetime#117, PULocationID#217L, trip_distance#157, fare_amount#277, tpep_pickup_datetime#97], functions=[first(VendorID#77L, false), first(passenger_count#1466, false), first(RatecodeID#177, false), first(store_and_fwd_flag#197, false), first(payment_type#257L, false), first(extra#297, false), first(mta_tax#317, false), first(tip_amount#337, false), first(tolls_amount#357, false), first(improvement_surcharge#377, false), first(total_amount#397, false), first(congestion_surcharge#417, false), first(airport_fee#437, false)])
   :  +- Sort [DOLocationID#237L ASC NULLS FIRST, tpep_dropoff_datetime#117 ASC NULLS FIRST, PULocationID#217L ASC NULLS FIRST, trip_distance#157 ASC NULLS FIRST, fare_amount#277 ASC NULLS FIRST, tpep_pickup_datetime#97 ASC NULLS FIRST], false, 0
   :    

In [3]:
# Explicit hint: force it, don't rely on the size estimate.
hinted_join = trips_clean.join(F.broadcast(zones), trips_clean.PULocationID == zones.LocationID)
hinted_join.explain()

# Look for "BroadcastHashJoin" and "BroadcastExchange" in the plan above --
# that confirms zones was broadcast, not shuffled. Compare against the
# default plan: if both show BroadcastHashJoin, Spark already made the same
# call automatically for this size of data.


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [PULocationID#217L], [cast(LocationID#1503 as bigint)], Inner, BuildRight, false
   :- SortAggregate(key=[DOLocationID#237L, tpep_dropoff_datetime#117, PULocationID#217L, trip_distance#157, fare_amount#277, tpep_pickup_datetime#97], functions=[first(VendorID#77L, false), first(passenger_count#1466, false), first(RatecodeID#177, false), first(store_and_fwd_flag#197, false), first(payment_type#257L, false), first(extra#297, false), first(mta_tax#317, false), first(tip_amount#337, false), first(tolls_amount#357, false), first(improvement_surcharge#377, false), first(total_amount#397, false), first(congestion_surcharge#417, false), first(airport_fee#437, false)])
   :  +- Sort [DOLocationID#237L ASC NULLS FIRST, tpep_dropoff_datetime#117 ASC NULLS FIRST, PULocationID#217L ASC NULLS FIRST, trip_distance#157 ASC NULLS FIRST, fare_amount#277 ASC NULLS FIRST, tpep_pickup_datetime#97 ASC NULLS FIRST], false, 0
   :    

## Join types quick reference

`df.join(other, condition, how=...)` — `how` values you'll actually use:

| `how` | Keeps |
|---|---|
| `"inner"` (default) | Only rows with a match on both sides |
| `"left"` | All left rows; unmatched right columns become `null` |
| `"left_semi"` | Left rows that *have* a match — right columns dropped entirely (a filter, not an enrichment) |
| `"left_anti"` | Left rows that *don't* have a match — useful for finding orphaned `LocationID`s |

Full list: [`DataFrame.join` API reference](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)

## Concept: join skew

**Skew** is when one join key has vastly more rows than others (e.g. one
`LocationID` — a busy Manhattan zone — appears millions of times while most
zones appear rarely). In a shuffle join, all rows for a key land on *one*
task/executor, so a skewed key creates one massive, slow task while everyone
else finishes early. Broadcast joins sidestep this entirely for the *small*
side, but if you ever shuffle-join two large skewed tables, this becomes a
real production problem — Spark's Adaptive Query Execution (AQE) has
skew-join handling for exactly this, which we'll cover in Phase 5
(performance tuning).

## Your task

1. **Join both zone columns.** `trips_clean` has both `PULocationID` and
   `DOLocationID`. Join `zones` **twice** — once for pickup, once for
   dropoff — and rename the resulting `Zone`/`Borough` columns so you end up
   with something like `pickup_zone`, `pickup_borough`, `dropoff_zone`,
   `dropoff_borough` alongside the original trip columns. (Hint: you'll hit
   ambiguous column name errors if you don't alias each `zones` reference
   distinctly — look up `DataFrame.alias()`.)

2. **Use `left`, not `inner`, for both joins**, and verify why: are there any
   `PULocationID`/`DOLocationID` values in `trips_clean` with no match in
   `zones`? (Hint: `left_anti` is the direct way to check this — don't just
   assume.)

3. **Confirm both joins broadcast.** Run `.explain()` on the final result and
   confirm you see two `BroadcastHashJoin` operators, not
   `SortMergeJoin`/`ShuffleExchange`.

4. **Aggregate something.** Once joined, produce the top 10 busiest pickup
   zones by trip count. This previews Phase 4 (aggregations), but it's also
   the real payoff of doing this join — raw `LocationID`s are meaningless on
   their own.


In [4]:
def join_zones(trips: DataFrame, zones: DataFrame) -> DataFrame:
    """Join pickup and dropoff LocationIDs to zone/borough names.

    Each `zones` reference is aliased ("pu"/"do") so the join condition and
    the later column selection can unambiguously refer to "which side" a
    Zone/Borough column came from -- without an alias, referencing
    `zones.Zone` twice after two joins is ambiguous to Spark's analyzer.

    We `.select()` immediately after each join instead of keeping every
    zones column (LocationID, service_zone) around -- that avoids duplicate
    LocationID columns colliding on the second join, and keeps the schema
    clean.
    """
    pu = zones.alias("pu")
    do = zones.alias("do")

    return (
        trips
        .join(pu, trips.PULocationID == pu.LocationID, "left")
        .select(
            *trips.columns,
            F.col("pu.Zone").alias("pickup_zone"),
            F.col("pu.Borough").alias("pickup_borough"),
        )
        .join(do, trips.DOLocationID == do.LocationID, "left")
        .select(
            *trips.columns,
            "pickup_zone",
            "pickup_borough",
            F.col("do.Zone").alias("dropoff_zone"),
            F.col("do.Borough").alias("dropoff_borough"),
        )
    )


trips_with_zones = join_zones(trips_clean, zones)

# --- Step 2: verify `left` vs `inner` actually matters here ---
orphaned_pu = trips_clean.join(zones, trips_clean.PULocationID == zones.LocationID, "left_anti").count()
orphaned_do = trips_clean.join(zones, trips_clean.DOLocationID == zones.LocationID, "left_anti").count()
print(f"Pickup IDs with no zone match: {orphaned_pu:,}")
print(f"Dropoff IDs with no zone match: {orphaned_do:,}")
# If either is > 0, `inner` would have silently dropped those trips --
# `left` preserves them (with null pickup_zone/dropoff_zone instead).

# --- Step 3: confirm both joins broadcast, not shuffle ---
trips_with_zones.explain()
# Look for two BroadcastHashJoin operators. If you instead see
# SortMergeJoin/ShuffleExchange, `zones` stopped being seen as "small enough"
# to auto-broadcast -- wrap it in F.broadcast(zones) inside join_zones to force it.

# --- Step 4: top 10 busiest pickup zones ---
trips_with_zones.groupBy("pickup_zone").count().orderBy(F.desc("count")).show(10, truncate=False)



Pickup IDs with no zone match: 0
Dropoff IDs with no zone match: 0
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [VendorID#2163L, tpep_pickup_datetime#97, tpep_dropoff_datetime#117, passenger_count#2165, trip_distance#157, RatecodeID#2167, store_and_fwd_flag#2169, PULocationID#217L, DOLocationID#237L, payment_type#2171L, fare_amount#277, extra#2173, mta_tax#2175, tip_amount#2177, tolls_amount#2179, improvement_surcharge#2181, total_amount#2183, congestion_surcharge#2185, airport_fee#2187, pickup_zone#1881, pickup_borough#1882, Zone#1906 AS dropoff_zone#1959, Borough#1905 AS dropoff_borough#1960]
   +- BroadcastHashJoin [DOLocationID#237L], [cast(LocationID#1904 as bigint)], LeftOuter, BuildRight, false
      :- Project [VendorID#2163L, tpep_pickup_datetime#97, tpep_dropoff_datetime#117, passenger_count#2165, trip_distance#157, RatecodeID#2167, store_and_fwd_flag#2169, PULocationID#217L, DOLocationID#237L, payment_type#2171L, fare_amount#277, extra#2173, mta_tax#217